# `aidp-atp` live test — IAM DB-Token + on-executor refresh

**Live-test row 5.** Long-running pattern. Pass criteria: zero auth failures over a >25-minute job (token refresh kicks in on each executor).


In [ ]:
import sys, os
# Adjust this if you've uploaded the plugin scripts/ dir elsewhere.
sys.path.insert(0, '/Workspace/Shared/oracle_ai_data_platform_connectors/scripts')


In [ ]:
from oracle_ai_data_platform_connectors.auth import generate_db_token
from oracle_ai_data_platform_connectors.auth.dbtoken import refresh_on_executors
from oracle_ai_data_platform_connectors.jdbc import build_oracle_jdbc_url, spark_jdbc_options_dbtoken

token_dir = generate_db_token(os.environ['ATP_COMPARTMENT_OCID'], target_dir='/tmp/dbcred_atp')
url = build_oracle_jdbc_url(
    tns_alias=os.environ['ATP_TNS_SERVICE'],
    tns_admin=os.environ.get('ATP_WALLET_PATH', '/tmp/wallet/atp'),
)
opts = spark_jdbc_options_dbtoken(url=url, token_dir=token_dir)


In [ ]:
# Driver-side query first (sanity check)
df = spark.read.format('jdbc').options(**opts).option('dbtable', os.environ['ATP_TABLE_FOR_TEST']).load()

# Then long-running mapPartitions with token refresh — for live tests, count rows
# per partition (cheap proxy for 'we held a JDBC connection long enough')
refresh = refresh_on_executors(spark, os.environ['ATP_COMPARTMENT_OCID'], '/tmp/dbcred_atp')
result = df.rdd.mapPartitions(lambda part: refresh(part)).toDF(df.schema)
print('result rows:', result.count())


In [ ]:
# Live-test driver parses this final cell's stdout for the JSON summary.
import json, time
summary = {
    'connector': 'aidp-atp',
    'auth': 'dbtoken',
    'rows': int(result.count()),
    'schema': sorted([f.name for f in result.schema.fields]),
    'timestamp_utc': int(time.time()),
}
print('AIDP_LIVE_TEST_RESULT_BEGIN')
print(json.dumps(summary, indent=2))
print('AIDP_LIVE_TEST_RESULT_END')
